<a href="https://colab.research.google.com/github/mablue/article.drugon/blob/main/Drugon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Cell 1: Environment Setup
# ============================================================
!pip install rdkit transformers datasets accelerate lightgbm scikit-learn torch -q

import os, pickle, warnings
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, TrainingArguments, Trainer
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, DataStructs
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ============================================================
# Cell 2: Data Loading
# ============================================================
!wget -q https://raw.githubusercontent.com/PatWalters/datafiles/main/CHEMBL4550.smi -O CHEMBL4550.smi

def parse_chembl(path):
    rows = []
    with open(path) as f:
        for line in f:
            p = line.strip().replace(",", " ").split()
            if len(p) < 3: continue
            smi = p[0]
            try: val = float(p[2])
            except ValueError: continue
            mol = Chem.MolFromSmiles(smi)
            if mol is None: continue
            if 15 <= mol.GetNumAtoms() <= 45:
                rows.append((smi, val))
    return rows

rows = parse_chembl("CHEMBL4550.smi")
smiles_list = [r[0] for r in rows]
y = np.array([r[1] for r in rows])
print(f"Dataset: {len(smiles_list)} molecules, pIC50 range {y.min():.2f}-{y.max():.2f}")

# ============================================================
# Cell 3: Phase Morgan Features (our method)
# ============================================================
def morgan_phase(smi, radius=2, n_bits=1024, mode="combined"):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return None
    from rdkit.Chem import rdMolDescriptors
    info = {}
    rdMolDescriptors.GetMorganFingerprint(mol, radius, bitInfo=info)
    arr = np.zeros(n_bits, dtype=np.complex64)
    for bit, entries in info.items():
        idx = bit % n_bits
        for atom_idx, r in entries:
            atom = mol.GetAtomWithIdx(atom_idx)
            if mode == "combined":
                phase = (atom.GetAtomicNum() * 0.15 +
                         (1.0 if atom.GetIsAromatic() else 0.0) * 1.5 +
                         atom.GetFormalCharge() * 1.2) * np.pi
            arr[idx] += np.exp(1j * phase)
    return np.concatenate([arr.real, arr.imag]).astype(np.float32)

print("Computing Phase Morgan features...")
X_phase = np.array([morgan_phase(s) for s in smiles_list])
print(f"Phase Morgan: {X_phase.shape}")

# ============================================================
# Cell 4: PyTorch Dataset for Transformers
# ============================================================
class SmilesDataset(Dataset):
    def __init__(self, smiles, labels, tokenizer, max_len=128):
        self.smiles = smiles
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.smiles)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.smiles[idx], truncation=True, padding="max_length",
                             max_length=self.max_len, return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
                "labels": torch.tensor(self.labels[idx], dtype=torch.float)}

# ============================================================
# Cell 5: ChemBERTa Fine-tuning
# ============================================================
class ChemBERTaRegressor(nn.Module):
    def __init__(self, model_name, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(hidden, 256), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(256, 1))
    def forward(self, input_ids, attention_mask, **kwargs):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0, :]  # CLS token
        return self.head(pooled).squeeze(-1)

def train_transformer(model_name, smiles_list, y, epochs=15, batch_size=32, lr=2e-5):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    dataset = SmilesDataset(smiles_list, y, tokenizer)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(y))
    for fold, (train_idx, val_idx) in enumerate(kf.split(smiles_list)):
        print(f"  Fold {fold+1}/5")
        train_ds = torch.utils.data.Subset(dataset, train_idx)
        val_ds = torch.utils.data.Subset(dataset, val_idx)
        model = ChemBERTaRegressor(model_name).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=batch_size)
        for epoch in range(epochs):
            model.train()
            for batch in train_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                optimizer.zero_grad()
                pred = model(batch["input_ids"], batch["attention_mask"])
                loss = nn.MSELoss()(pred, batch["labels"])
                loss.backward(); optimizer.step()
            model.eval()
            preds = []
            with torch.no_grad():
                for batch in val_loader:
                    batch = {k: v.to(device) for k, v in batch.items()}
                    preds.append(model(batch["input_ids"], batch["attention_mask"]).cpu().numpy())
            oof_preds[val_idx] = np.concatenate(preds)
            if (epoch+1) % 5 == 0:
                r = pearsonr(y[val_idx], oof_preds[val_idx])[0]
                print(f"    Epoch {epoch+1}: r={r:.3f}")
    r = pearsonr(y, oof_preds)[0]
    rho = spearmanr(y, oof_preds)[0]
    rmse = np.sqrt(mean_squared_error(y, oof_preds))
    print(f"  ChemBERTa FINAL: r={r:.3f}  rho={rho:.3f}  rmse={rmse:.3f}")
    return r, rho, rmse

print("=" * 60)
print("ChemBERTa Fine-tuning")
print("=" * 60)
r_chemberta, rho_cb, rmse_cb = train_transformer(
    "seyonec/ChemBERTa-zinc-base-v1", smiles_list, y, epochs=15)

# ============================================================
# Cell 6: MolFormer Fine-tuning
# ============================================================
print("=" * 60)
print("MolFormer Fine-tuning")
print("=" * 60)
r_molformer, rho_mf, rmse_mf = train_transformer(
    "ibm/MoLFormer-XL-both-10pct", smiles_list, y, epochs=15)

# ============================================================
# Cell 7: Phase Morgan + LightGBM
# ============================================================
print("=" * 60)
print("Phase Morgan + LightGBM")
print("=" * 60)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_phase = np.zeros(len(y))
for train_idx, val_idx in kf.split(X_phase):
    model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, num_leaves=31,
                               max_depth=7, min_child_samples=5, subsample=0.8,
                               colsample_bytree=0.7, random_state=42, n_jobs=-1, verbose=-1)
    model.fit(X_phase[train_idx], y[train_idx])
    oof_phase[val_idx] = model.predict(X_phase[val_idx])
r_phase = pearsonr(y, oof_phase)[0]
rho_p = spearmanr(y, oof_phase)[0]
rmse_p = np.sqrt(mean_squared_error(y, oof_phase))
print(f"  Phase Morgan FINAL: r={r_phase:.3f}  rho={rho_p:.3f}  rmse={rmse_p:.3f}")

# ============================================================
# Cell 8: Final Comparison
# ============================================================
print("\n" + "=" * 60)
print("FINAL COMPARISON ON CHEMBL c-Abl pIC50")
print("=" * 60)
print(f"{'Method':<25} {'r':>8} {'rho':>8} {'rmse':>8}")
print("-" * 60)
print(f"{'Phase Morgan + LGBM':<25} {r_phase:>+8.3f} {rho_p:>+8.3f} {rmse_p:>8.3f}")
print(f"{'ChemBERTa (fine-tuned)':<25} {r_chemberta:>+8.3f} {rho_cb:>+8.3f} {rmse_cb:>8.3f}")
print(f"{'MolFormer (fine-tuned)':<25} {r_molformer:>+8.3f} {rho_mf:>+8.3f} {rmse_mf:>8.3f}")
print("-" * 60)
print(f"Dataset: {len(smiles_list)} molecules, 5-fold CV")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 54.8 MB/s eta 0:00:00
Device: cuda
Dataset: 1378 molecules, pIC50 range 5.60-9.47
Computing Phase Morgan features...
Phase Morgan: (1378, 2048)
ChemBERTa Fine-tuning


config.json:   0%|          | 0.00/501 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/9.43k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.21k [00:00<?, ?B/s]

...

[transformers] You are using a model of type `molformer` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


tokenizer_config.json:   0%|          | 0.00/1.13k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/54.0k [00:00<?, ?B/s]

  Fold 1/5
The repository ibm/MoLFormer-XL-both-10pct contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/ibm/MoLFormer-XL-both-10pct .
 You can inspect the repository content at https://hf.co/ibm/MoLFormer-XL-both-10pct.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] 


ValueError: The repository ibm/MoLFormer-XL-both-10pct contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/ibm/MoLFormer-XL-both-10pct .
 You can inspect the repository content at https://hf.co/ibm/MoLFormer-XL-both-10pct.
Please pass the argument `trust_remote_code=True` to allow custom code to be run.

In [2]:
# ============================================================
# Cell Fix: MolFormer with trust_remote_code + Phase Morgan final
# ============================================================
from transformers import AutoConfig, AutoModel, AutoTokenizer

# --- Fix 1: MolFormer loader with trust_remote_code ---
class MoLFormerRegressor(nn.Module):
    def __init__(self, model_name, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(
            model_name, trust_remote_code=True)
        hidden = self.encoder.config.hidden_size
        self.head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(hidden, 256), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(256, 1))
    def forward(self, input_ids, attention_mask, **kwargs):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0, :]
        return self.head(pooled).squeeze(-1)


def train_molformer(smiles_list, y, epochs=15, batch_size=32, lr=2e-5):
    model_name = "ibm/MoLFormer-XL-both-10pct"
    tokenizer = AutoTokenizer.from_pretrained(
        model_name, trust_remote_code=True)
    dataset = SmilesDataset(smiles_list, y, tokenizer)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros(len(y))
    for fold, (tr, va) in enumerate(kf.split(smiles_list)):
        print(f"  Fold {fold+1}/5")
        train_ds = torch.utils.data.Subset(dataset, tr)
        val_ds = torch.utils.data.Subset(dataset, va)
        model = MoLFormerRegressor(model_name).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr)
        tl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        vl = DataLoader(val_ds, batch_size=batch_size)
        for ep in range(epochs):
            model.train()
            for b in tl:
                b = {k: v.to(device) for k, v in b.items()}
                opt.zero_grad()
                loss = nn.MSELoss()(model(b["input_ids"], b["attention_mask"]),
                                     b["labels"])
                loss.backward(); opt.step()
            model.eval()
            preds = []
            with torch.no_grad():
                for b in vl:
                    b = {k: v.to(device) for k, v in b.items()}
                    preds.append(model(b["input_ids"],
                                        b["attention_mask"]).cpu().numpy())
            oof[va] = np.concatenate(preds)
            if (ep+1) % 5 == 0:
                print(f"    Epoch {ep+1}: r={pearsonr(y[va], oof[va])[0]:.3f}")
    r = pearsonr(y, oof)[0]
    rho = spearmanr(y, oof)[0]
    rmse = np.sqrt(mean_squared_error(y, oof))
    print(f"  MoLFormer FINAL: r={r:.3f}  rho={rho:.3f}  rmse={rmse:.3f}")
    return r, rho, rmse


print("=" * 60)
print("MoLFormer Fine-tuning")
print("=" * 60)
try:
    r_molformer, rho_mf, rmse_mf = train_molformer(smiles_list, y, epochs=15)
except Exception as e:
    print(f"MoLFormer failed: {e}")
    r_molformer, rho_mf, rmse_mf = 0.0, 0.0, 0.0


# --- Phase Morgan + LightGBM (final) ---
print("\n" + "=" * 60)
print("Phase Morgan + LightGBM")
print("=" * 60)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_phase = np.zeros(len(y))
for tr, va in kf.split(X_phase):
    m = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03,
                           num_leaves=31, max_depth=7, min_child_samples=5,
                           subsample=0.8, colsample_bytree=0.7,
                           random_state=42, n_jobs=-1, verbose=-1)
    m.fit(X_phase[tr], y[tr])
    oof_phase[va] = m.predict(X_phase[va])
r_phase = pearsonr(y, oof_phase)[0]
rho_p = spearmanr(y, oof_phase)[0]
rmse_p = np.sqrt(mean_squared_error(y, oof_phase))
print(f"  Phase Morgan FINAL: r={r_phase:.3f}  rho={rho_p:.3f}  rmse={rmse_p:.3f}")


# --- FINAL COMPARISON ---
print("\n" + "=" * 60)
print("FINAL COMPARISON — ChEMBL c-Abl (1378 molecules, 5-fold CV)")
print("=" * 60)
print(f"{'Method':<28} {'r':>8} {'rho':>8} {'rmse':>8}")
print("-" * 60)
print(f"{'Phase Morgan + LGBM':<28} {r_phase:>+8.3f} {rho_p:>+8.3f} {rmse_p:>8.3f}")
print(f"{'ChemBERTa (fine-tuned)':<28} {r_chemberta:>+8.3f} {rho_cb:>+8.3f} {rmse_cb:>8.3f}")
print(f"{'MoLFormer (fine-tuned)':<28} {r_molformer:>+8.3f} {rho_mf:>+8.3f} {rmse_mf:>8.3f}")
print("-" * 60)

MoLFormer Fine-tuning


configuration_molformer.py:   0%|          | 0.00/7.10k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/ibm/MoLFormer-XL-both-10pct:
- configuration_molformer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


  Fold 1/5


modeling_molformer.py:   0%|          | 0.00/36.9k [00:00<?, ?B/s]

...

[transformers] MolformerModel LOAD REPORT from: ibm/MoLFormer-XL-both-10pct
Key                                | Status     |  | 
-----------------------------------+------------+--+-
lm_head.transform.LayerNorm.weight | UNEXPECTED |  | 
lm_head.transform.LayerNorm.bias   | UNEXPECTED |  | 
lm_head.decoder.weight             | UNEXPECTED |  | 
lm_head.transform.dense.bias       | UNEXPECTED |  | 
lm_head.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    Epoch 5: r=0.370
    Epoch 10: r=0.454
    Epoch 15: r=0.580
  Fold 5/5


Loading weights:   0%|          | 0/207 [00:00<?, ?it/s]

[transformers] MolformerModel LOAD REPORT from: ibm/MoLFormer-XL-both-10pct
Key                                | Status     |  | 
-----------------------------------+------------+--+-
lm_head.transform.LayerNorm.weight | UNEXPECTED |  | 
lm_head.transform.LayerNorm.bias   | UNEXPECTED |  | 
lm_head.decoder.weight             | UNEXPECTED |  | 
lm_head.transform.dense.bias       | UNEXPECTED |  | 
lm_head.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    Epoch 5: r=0.282
    Epoch 10: r=0.284
    Epoch 15: r=0.548
  MoLFormer FINAL: r=0.551  rho=0.553  rmse=0.906

Phase Morgan + LightGBM
  Phase Morgan FINAL: r=0.749  rho=0.753  rmse=0.613

FINAL COMPARISON — ChEMBL c-Abl (1378 molecules, 5-fold CV)
Method                              r      rho     rmse
------------------------------------------------------------
Phase Morgan + LGBM            +0.749   +0.753    0.613
ChemBERTa (fine-tuned)         +0.605   +0.623    0.859
MoLFormer (fine-tuned)         +0.551   +0.553    0.906
------------------------------------------------------------


In [3]:
# ============================================================
# Step 1: Download two independent targets from ChEMBL
# ============================================================
!pip install chembl_webresource_client -q

from chembl_webresource_client.new_client import new_client
import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")

def download_target(target_id, name, max_n=2000):
    """Download and clean ChEMBL binding activities for one target."""
    print(f"\nDownloading {name} ({target_id})...")
    activities = new_client.activity.filter(
        target_chembl_id=target_id,
        assay_type='B',
        pchembl_value__gte=4.0,
        pchembl_value__lte=11.0,
    ).only(['canonical_smiles', 'pchembl_value'])

    rows, seen = [], set()
    for a in activities:
        smi = a.get('canonical_smiles')
        pval = a.get('pchembl_value')
        if not smi or pval is None:
            continue
        try:
            pval = float(pval)
        except (ValueError, TypeError):
            continue
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        n = mol.GetNumAtoms()
        if not (15 <= n <= 50):
            continue
        if smi in seen:
            continue
        seen.add(smi)
        rows.append({'smiles': smi, 'pic50': pval})
        if len(rows) >= max_n:
            break

    df = pd.DataFrame(rows)
    print(f"  {len(df)} unique molecules")
    return df


df_ache = download_target('CHEMBL220', 'AChE')
df_ache.to_csv('ache.csv', index=False)

df_cox2 = download_target('CHEMBL230', 'COX-2')
df_cox2.to_csv('cox2.csv', index=False)

print("\n✓ Downloaded and saved.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 4.3 MB/s eta 0:00:00

  2000 unique molecules

  2000 unique molecules

✓ Downloaded and saved.


In [4]:

# ============================================================
# Step 2: Phase Morgan features + LightGBM + baseline
# ============================================================
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr, spearmanr
from rdkit.Chem import AllChem, DataStructs, rdMolDescriptors

def morgan_binary(smi, radius=2, n_bits=1024):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return None
    gen = AllChem.GetMorganGenerator(radius=radius, fpSize=n_bits)
    fp = gen.GetFingerprint(mol)
    arr = np.zeros(n_bits, dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def morgan_phase(smi, radius=2, n_bits=1024):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return None
    info = {}
    rdMolDescriptors.GetMorganFingerprint(mol, radius, bitInfo=info)
    arr = np.zeros(n_bits, dtype=np.complex64)
    for bit, entries in info.items():
        idx = bit % n_bits
        for atom_idx, r in entries:
            atom = mol.GetAtomWithIdx(atom_idx)
            phase = (atom.GetAtomicNum() * 0.15 +
                     (1.0 if atom.GetIsAromatic() else 0.0) * 1.5 +
                     atom.GetFormalCharge() * 1.2) * np.pi
            arr[idx] += np.exp(1j * phase)
    return np.concatenate([arr.real, arr.imag]).astype(np.float32)


def evaluate_target(df, name):
    print(f"\n{'='*60}")
    print(f"Target: {name}  ({len(df)} molecules)")
    print(f"{'='*60}")

    smiles = df['smiles'].tolist()
    y = df['pic50'].values
    print(f"  pIC50 range: {y.min():.2f} - {y.max():.2f}  (mean {y.mean():.2f})")

    # Compute features
    print("  Computing features...")
    X_bin = np.array([morgan_binary(s) for s in smiles])
    X_ph = np.array([morgan_phase(s) for s in smiles])

    # CV
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    def cv_score(X):
        oof = np.zeros(len(y))
        for tr, va in kf.split(X):
            m = lgb.LGBMRegressor(
                n_estimators=500, learning_rate=0.03, num_leaves=31,
                max_depth=7, min_child_samples=5, subsample=0.8,
                colsample_bytree=0.7, random_state=42, n_jobs=-1, verbose=-1)
            m.fit(X[tr], y[tr])
            oof[va] = m.predict(X[va])
        r = pearsonr(y, oof)[0]
        rho = spearmanr(y, oof)[0]
        rmse = np.sqrt(mean_squared_error(y, oof))
        return r, rho, rmse

    r_bin, rho_b, rmse_b = cv_score(X_bin)
    r_ph, rho_p, rmse_p = cv_score(X_ph)

    print(f"\n  {'Method':<22} {'r':>8}  {'rho':>8}  {'rmse':>8}")
    print(f"  {'-'*22}  {'-'*8}  {'-'*8}  {'-'*8}")
    print(f"  {'Binary Morgan':<22} {r_bin:>+8.3f}  {rho_b:>+8.3f}  {rmse_b:>8.3f}")
    print(f"  {'Phase Morgan':<22} {r_ph:>+8.3f}  {rho_p:>+8.3f}  {rmse_p:>8.3f}")

    delta = r_ph - r_bin
    if delta > 0.01:
        print(f"\n  → Phase Morgan IMPROVED by {delta:+.3f}")
    elif delta > -0.01:
        print(f"\n  → Tied (delta {delta:+.3f})")
    else:
        print(f"\n  → Binary won by {-delta:.3f}")

    return {'name': name, 'n': len(df), 'r_bin': r_bin, 'r_ph': r_ph,
            'delta': delta}


results = []
results.append(evaluate_target(df_ache, 'AChE (CHEMBL220)'))
results.append(evaluate_target(df_cox2, 'COX-2 (CHEMBL230)'))

# ============================================================
# Step 3: Final summary table
# ============================================================
print(f"\n{'='*60}")
print("CROSS-TARGET SUMMARY")
print(f"{'='*60}")
print(f"{'Target':<20} {'n':>6}  {'Binary':>8}  {'Phase':>8}  {'Δ':>7}")
print(f"{'-'*20}  {'-'*6}  {'-'*8}  {'-'*8}  {'-'*7}")

# Add c-Abl reference (from v20)
print(f"{'c-Abl (CHEMBL4550)':<20} {1378:>6}  {0.723:>+8.3f}  {0.749:>+8.3f}  {0.026:>+7.3f}")
for r in results:
    print(f"{r['name']:<20} {r['n']:>6}  {r['r_bin']:>+8.3f}  {r['r_ph']:>+8.3f}  {r['delta']:>+7.3f}")


Target: AChE (CHEMBL220)  (2000 molecules)
  pIC50 range: 4.00 - 10.93  (mean 6.35)
  Computing features...

  Method                        r       rho      rmse
  ----------------------  --------  --------  --------
  Binary Morgan            +0.834    +0.831     0.765
  Phase Morgan             +0.844    +0.838     0.746

  → Tied (delta +0.010)

Target: COX-2 (CHEMBL230)  (2000 molecules)
  pIC50 range: 4.00 - 10.70  (mean 6.47)
  Computing features...

  Method                        r       rho      rmse
  ----------------------  --------  --------  --------
  Binary Morgan            +0.741    +0.738     0.819
  Phase Morgan             +0.742    +0.740     0.819

  → Tied (delta +0.000)

CROSS-TARGET SUMMARY
Target                    n    Binary     Phase        Δ
--------------------  ------  --------  --------  -------
c-Abl (CHEMBL4550)     1378    +0.723    +0.749   +0.026
AChE (CHEMBL220)       2000    +0.834    +0.844   +0.010
COX-2 (CHEMBL230)      2000    +0.741    +0.